In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from ffm.posthog_api import run_hogql, get_feature_flags
run_hogql("SELECT event, count() AS n FROM events GROUP BY event ORDER BY n DESC")

,event,n
0,$pageview,4454
1,$screen,1312
2,order_completed,1162
3,$feature_flag_called,425
4,$exception,352
5,$identify,61


In [2]:
import json
from ffm.diagnose import diagnose
tickets = json.load(open("../tickets.json"))
expected = {"T1": "C1", "T2": "C5", "T3": "C4", "T4": "C6", "T5": "C2"}  # C2 (old SDK) is the cause, C3 (no URL) the symptom
for t in tickets:
    d = diagnose(t)
    top = d["results"][0].id
    print(t["id"], "top:", top, "OK" if top == expected[t["id"]] else "CHECK", f'{d["seconds"]:.1f}s')
print(diagnose(tickets[0])["note_markdown"])

T1 top: C1 OK 7.0s
T2 top: C5 OK 2.0s
T3 top: C4 OK 2.1s
T4 top: C6 OK 2.1s
T5 top: C2 OK 2.0s
## T1: Mobile data gone?

**Area:** ingestion | **Team:** Ingestion | **Severity:** high

### Likely cause

- **C1 SDK went quiet:** posthog-react-native stopped while other SDKs kept sending, which points to that app or service rather than the whole project. A recent release may have removed or broken the SDK.

### Evidence

**C1 SDK went quiet (fail):** posthog-react-native has sent nothing for 73 hours, while posthog-python kept sending

| lib | last_seen | hours_silent | hours_behind_newest | status |
| --- | --- | --- | --- | --- |
| posthog-react-native | 2026-10-31T01:39:11.639000Z | 73 | 73 | fail |
| posthog-python | 2026-11-03T02:07:37.255000Z | 0 | 0 | pass |
| posthog-js | 2026-11-03T02:01:23.693000Z | 0 | 0 | pass |

**C8 Overall volume drop (pass):** 781 events in the last 24 hours, 138% of the 7-day daily average of 564

| last_24h | daily_average | pct_of_average |
| --- | ---

In [2]:
import json
from ffm.classify import classify
tickets = json.load(open("../tickets.json"))
for t in tickets:
    print(t["id"], classify(t))

T1 {'area': 'ingestion', 'team': 'Ingestion', 'severity': 'high', 'flag_key': None, 'keywords': ['gone', 'flat']}
T2 {'area': 'feature_flags', 'team': 'Feature Flags', 'severity': 'medium', 'flag_key': 'new-checkout', 'keywords': ['flag']}
T3 {'area': 'identification', 'team': 'Product Analytics', 'severity': 'low', 'flag_key': None, 'keywords': ['unique users', 'users']}
T4 {'area': 'error_tracking', 'team': 'Error Tracking', 'severity': 'high', 'flag_key': None, 'keywords': ['error', 'spike']}
T5 {'area': 'data_quality', 'team': 'Client Libraries (SDKs)', 'severity': 'low', 'flag_key': None, 'keywords': ['url', 'blank', 'filter']}


In [3]:
from ffm import checks

for fn in checks.ALL_CHECKS:
    r = fn()
    print(r.id, r.status, "|", r.headline)

C1 fail | posthog-react-native has sent nothing for 73 hours, while posthog-python kept sending
C2 fail | posthog-js 1.36.0 is 409 minor releases behind the latest 1.445.1, and sent 430 events in 7 days
C3 fail | 430 of 2,180 pageviews (19.7%) in 7 days have no $current_url, from SDK version 1.36.0
C4 fail | posthog-js has 4,796 distinct IDs and 0 $identify calls in 14 days (1 more SDK also flagged)
C5 fail | new-checkout is active, but 0 of 4,515 persons match email icontains @tidewater.io, and all 425 $feature_flag_called events in 7 days returned false
C6 fail | 280 errors in the last 24 hours, 50.6x the daily average of 5.5 from the 13 days before. Most common: "TypeError: Cannot read properties of undefined (reading 'total')" on app_version 2.4.0 (280)
C7 fail | 286 of 572 order_completed rows (50.0%) in 7 days are exact duplicates
C8 pass | 781 events in the last 24 hours, 138% of the 7-day daily average of 564


In [3]:
from ffm import checks

for fn in [checks.check_c1_sdk_quiet, checks.check_c2_sdk_versions, checks.check_c3_missing_properties, checks.check_c4_identification]:
    r = fn()
    print(r.id, r.status, "|", r.headline)

C1 fail | posthog-react-native has sent nothing for 73 hours, while posthog-python kept sending
C2 fail | posthog-js 1.36.0 is 409 minor releases behind the latest 1.445.1, and sent 430 events in 7 days
C3 fail | 430 of 2,180 pageviews (19.7%) in 7 days have no $current_url, from SDK version 1.36.0
C4 fail | posthog-js has 4,796 distinct IDs and 0 $identify calls in 14 days (1 more SDK also flagged)


In [2]:
[(f["key"], f["active"]) for f in get_feature_flags()]

[('new-checkout', True)]